# Telemedicine Operations Intelligence Platform
## Databricks PySpark ETL, Analytics, and Machine Learning Forecasting

**Purpose:** This notebook builds a scalable ETL and ML workflow using Databricks and PySpark. It reads the raw telehealth dataset, cleans and transforms the data, creates analytics-ready tables, compares forecasting models, and generates decision-support outputs for healthcare, business, and policy audiences.

## 1. Load Raw Data
The raw CSV file is stored in a Databricks Unity Catalog volume. This step represents data ingestion into the Databricks environment.

In [ ]:
from pyspark.sql.functions import (
    col, avg, count, max as spark_max, min as spark_min,
    when, row_number, desc, asc
)
from pyspark.sql.window import Window

RAW_PATH = "/Volumes/workspace/default/telehealth_volume/telehealth_data.csv"

df = spark.read.csv(
    RAW_PATH,
    header=True,
    inferSchema=True
)

# Standardize column names to lowercase snake_case
for c in df.columns:
    clean_name = c.strip().lower().replace(" ", "_")
    df = df.withColumnRenamed(c, clean_name)

display(df.limit(10))
df.printSchema()

## 2. ETL Processing
This step cleans and prepares the dataset for analytics. The transformation filters the dataset to overall yearly records and converts the telehealth rate into a percentage.

In [ ]:
processed_df = (
    df
    .filter(col("quarter") == "Overall")
    .withColumn("year", col("year").cast("int"))
    .withColumn("total_enrollment", col("total_enrollment").cast("double"))
    .withColumn("telehealth_users", col("telehealth_users").cast("double"))
    .withColumn("telehealth_rate", col("telehealth_rate").cast("double") * 100)
)

# Remove rows with missing critical modeling fields
processed_df = processed_df.dropna(
    subset=["year", "region", "total_enrollment", "telehealth_users", "telehealth_rate"]
)

display(processed_df.limit(10))

## 3. Save Processed Table
The cleaned dataset is saved as a Databricks table. This demonstrates the load step of the ETL pipeline and creates an analytics-ready table.

In [ ]:
processed_df.write.mode("overwrite").saveAsTable("default.telehealth_processed")

In [ ]:
%sql
SELECT * FROM default.telehealth_processed LIMIT 10;

## 4. KPI Summary
These indicators provide a quick business and healthcare overview of the processed dataset.

In [ ]:
kpi_df = processed_df.agg(
    count("*").alias("records"),
    avg("telehealth_rate").alias("avg_telehealth_rate"),
    spark_max("telehealth_rate").alias("max_telehealth_rate"),
    spark_min("telehealth_rate").alias("min_telehealth_rate")
)

display(kpi_df)

## 5. National Telehealth Adoption Trend
This shows how average telehealth adoption changed over time.

**Recommended visualization:** Line chart  
- X-axis: `year`  
- Y-axis: `avg_telehealth_rate`

In [ ]:
yearly_trend = (
    processed_df
    .groupBy("year")
    .agg(avg("telehealth_rate").alias("avg_telehealth_rate"))
    .orderBy("year")
)

display(yearly_trend)

## 6. Regional Performance Comparison
These views identify the highest and lowest adoption regions in the latest available year.

In [ ]:
latest_year = int(processed_df.agg(spark_max("year")).collect()[0][0])

top_regions = (
    processed_df
    .filter(col("year") == latest_year)
    .groupBy("region")
    .agg(avg("telehealth_rate").alias("avg_telehealth_rate"))
    .orderBy(col("avg_telehealth_rate").desc())
    .limit(10)
)

display(top_regions)

In [ ]:
low_regions = (
    processed_df
    .filter(col("year") == latest_year)
    .groupBy("region")
    .agg(avg("telehealth_rate").alias("avg_telehealth_rate"))
    .orderBy(col("avg_telehealth_rate").asc())
    .limit(10)
)

display(low_regions)

## 7. Adoption Segmentation
Regions are grouped into adoption categories to support policy and business decision-making.

- High Adoption: 50% or higher  
- Moderate Adoption: 25% to 49.99%  
- Low Adoption: below 25%

In [ ]:
segment_df = (
    processed_df
    .groupBy("region")
    .agg(avg("telehealth_rate").alias("avg_telehealth_rate"))
    .withColumn(
        "segment",
        when(col("avg_telehealth_rate") >= 50, "High Adoption")
        .when(col("avg_telehealth_rate") >= 25, "Moderate Adoption")
        .otherwise("Low Adoption")
    )
)

segment_count = (
    segment_df
    .groupBy("segment")
    .agg(count("*").alias("number_of_regions"))
)

display(segment_count)

## 8. Telehealth Adoption Gap Over Time
This measures inequality between the top 10 and lowest 10 regions each year.

In [ ]:
region_year = (
    processed_df
    .groupBy("year", "region")
    .agg(avg("telehealth_rate").alias("avg_rate"))
)

top_window = Window.partitionBy("year").orderBy(desc("avg_rate"))
low_window = Window.partitionBy("year").orderBy(asc("avg_rate"))

top_10 = (
    region_year
    .withColumn("rank", row_number().over(top_window))
    .filter(col("rank") <= 10)
    .groupBy("year")
    .agg(avg("avg_rate").alias("top_10_avg"))
)

low_10 = (
    region_year
    .withColumn("rank", row_number().over(low_window))
    .filter(col("rank") <= 10)
    .groupBy("year")
    .agg(avg("avg_rate").alias("low_10_avg"))
)

gap_trend = (
    top_10
    .join(low_10, on="year")
    .withColumn("adoption_gap", col("top_10_avg") - col("low_10_avg"))
    .orderBy("year")
)

display(gap_trend)

## 9. Machine Learning Dataset
The model uses the available dataset variables:

- `year` = time dimension  
- `region` = geography  
- `total_enrollment` = market size  
- `telehealth_users` = usage behavior  
- `telehealth_rate` = prediction target

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ml_df = (
    processed_df
    .groupBy("year", "region")
    .agg(
        avg("telehealth_rate").alias("telehealth_rate"),
        avg("total_enrollment").alias("total_enrollment"),
        avg("telehealth_users").alias("telehealth_users")
    )
)

display(ml_df.limit(10))

## 10. Encode Region and Build Feature Vector
Spark ML requires features to be converted into a numeric vector. The region variable is encoded using `StringIndexer`.

In [ ]:
indexer = StringIndexer(
    inputCol="region",
    outputCol="region_index",
    handleInvalid="keep"
)

indexer_model = indexer.fit(ml_df)
ml_indexed = indexer_model.transform(ml_df)

assembler = VectorAssembler(
    inputCols=["year", "region_index", "total_enrollment", "telehealth_users"],
    outputCol="features",
    handleInvalid="skip"
)

ml_ready = assembler.transform(ml_indexed).select(
    "year",
    "region",
    "region_index",
    "total_enrollment",
    "telehealth_users",
    "features",
    "telehealth_rate"
)

display(ml_ready.limit(10))

## 11. Train/Test Split
The dataset is split into training and testing sets to evaluate model performance on unseen data.

**Recommended visualization:** Bar chart  
- X-axis: `dataset`  
- Y-axis: `records`

In [ ]:
train_df, test_df = ml_ready.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()
total_count = train_count + test_count

split_summary = spark.createDataFrame([
    ("Training Set", train_count, round((train_count / total_count) * 100, 2)),
    ("Testing Set", test_count, round((test_count / total_count) * 100, 2))
], ["dataset", "records", "percentage"])

display(split_summary)

## 12. Train Forecasting Models
Two models are trained:

1. Linear Regression — baseline model  
2. Random Forest Regressor — machine learning model that can capture non-linear patterns

In [ ]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="telehealth_rate"
)

lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

display(lr_predictions.select("year", "region", "telehealth_rate", "prediction"))

In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="telehealth_rate",
    numTrees=100,
    maxDepth=5,
    maxBins=200,
    seed=42
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)

display(rf_predictions.select("year", "region", "telehealth_rate", "prediction"))

## 13. Model Performance Comparison
Models are evaluated using Mean Absolute Error (MAE). Lower MAE means better predictive performance.

**Recommended visualization:** Bar chart  
- X-axis: `model`  
- Y-axis: `mae`

In [ ]:
evaluator = RegressionEvaluator(
    labelCol="telehealth_rate",
    predictionCol="prediction",
    metricName="mae"
)

lr_mae = evaluator.evaluate(lr_predictions)
rf_mae = evaluator.evaluate(rf_predictions)

model_results = spark.createDataFrame([
    ("Linear Regression", float(lr_mae)),
    ("Random Forest Regressor", float(rf_mae))
], ["model", "mae"])

display(model_results)

best_model = rf_model if rf_mae < lr_mae else lr_model
best_model_name = "Random Forest Regressor" if rf_mae < lr_mae else "Linear Regression"

print(f"Best Model: {best_model_name}")
print(f"Linear Regression MAE: {lr_mae:.2f}")
print(f"Random Forest MAE: {rf_mae:.2f}")

## 14. Actual vs Predicted Values
This supports model validation by comparing real values with model predictions.

**Recommended visualization:** Scatter plot  
- X-axis: `telehealth_rate`  
- Y-axis: `prediction`

In [ ]:
actual_vs_predicted = rf_predictions.select(
    "year",
    "region",
    "telehealth_rate",
    "prediction"
)

display(actual_vs_predicted)

## 15. Future Forecast Dataset
For future years, the model uses each region's latest known enrollment and telehealth users instead of zeros. This keeps the forecast realistic.

In [ ]:
latest_year = int(ml_ready.agg(spark_max("year")).collect()[0][0])
future_years = [latest_year + 1, latest_year + 2, latest_year + 3]

latest_window = Window.partitionBy("region").orderBy(desc("year"))

latest_region_features = (
    ml_indexed
    .withColumn("rn", row_number().over(latest_window))
    .filter(col("rn") == 1)
    .select("region", "region_index", "total_enrollment", "telehealth_users")
)

future_years_df = spark.createDataFrame(
    [(int(y),) for y in future_years],
    ["year"]
)

future_df = latest_region_features.crossJoin(future_years_df)

future_ready = assembler.transform(future_df)

future_predictions = best_model.transform(future_ready)

future_predictions = future_predictions.withColumn(
    "forecasted_telehealth_rate",
    when(col("prediction") < 0, 0)
    .when(col("prediction") > 100, 100)
    .otherwise(col("prediction"))
)

display(
    future_predictions.select(
        "year",
        "region",
        "forecasted_telehealth_rate"
    )
)

## 16. National Forecast and Adoption Level
Regional forecasts are aggregated to create a national forecast. The forecast is also classified into adoption levels for easier interpretation by healthcare and business stakeholders.

In [ ]:
national_forecast = (
    future_predictions
    .groupBy("year")
    .agg(avg("forecasted_telehealth_rate").alias("forecasted_telehealth_rate"))
    .orderBy("year")
)

national_forecast = national_forecast.withColumn(
    "adoption_level",
    when(col("forecasted_telehealth_rate") >= 50, "High")
    .when(col("forecasted_telehealth_rate") >= 25, "Moderate")
    .otherwise("Low")
)

display(national_forecast)

## 17. Actual Trend and Forecast Comparison
This combines historical adoption rates with future model-based forecasts.

**Recommended visualization:** Line chart  
- X-axis: `year`  
- Y-axis: `actual_rate` and `forecasted_telehealth_rate`

In [ ]:
actual_trend = (
    processed_df
    .groupBy("year")
    .agg(avg("telehealth_rate").alias("actual_rate"))
)

combined = (
    actual_trend
    .join(national_forecast, on="year", how="outer")
    .orderBy("year")
)

display(combined)

last_row = national_forecast.orderBy(desc("year")).first()
last_year = last_row["year"]
last_value = last_row["forecasted_telehealth_rate"]
last_level = last_row["adoption_level"]

print(f"""
Forecast Insight:
- Telehealth adoption is projected to reach approximately {last_value:.2f}% by {last_year}.
- The forecasted national adoption level is classified as {last_level}.
- This forecast should be interpreted as a prototype decision-support estimate, not a final operational forecast.
""")

## 18. Project Notes 

**What was implemented:**  
- Databricks Unity Catalog volume for file storage  
- PySpark ETL pipeline for distributed data processing  
- Processed Databricks table for analytics  
- KPI, trend, regional ranking, segmentation, and gap analysis  
- Machine learning forecasting using Linear Regression and Random Forest  
- Model evaluation using MAE  

**limitation:**  
Although the dataset contains many observations, the forecast is based on a limited number of historical years. Future work should include higher-frequency data, broadband access, provider availability, reimbursement policy variables, and demographic features.